# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Mapa leído de la imagen (filas 0-4 de arriba a abajo, columnas 0-5 de izquierda a derecha)
        self.start = (0, 0)

        # Estanterías (celdas grises con estante)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso (celdas amarillas tipo panal)
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Terminales: entrega +10, estación de carga +2, peligro mortal -10
        self.terminal_states = {
            (0, 5): +10.0,   # ENTREGA
            (2, 2): +2.0,    # CARGA
            (3, 5): -10.0,   # PELIGRO MORTAL
        }

        # Peligros no terminales (-3)
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        # Probabilidad de moverse en la dirección elegida; el resto se reparte
        # a partes iguales entre las dos desviaciones (izquierda y derecha).
        self.p_intended_normal = 0.90      # piso normal: 0.90 / 0.05 / 0.05
        self.p_intended_slippery = 0.60    # piso resbaloso: 0.60 / 0.20 / 0.20

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): recompensa del estado actual.
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Un estado terminal es absorbente.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Las probabilidades dependen del piso del estado ACTUAL.
        if state in self.slippery_states:
            p_intended = self.p_intended_slippery
        else:
            p_intended = self.p_intended_normal
        p_side = (1.0 - p_intended) / 2.0

        # Desviaciones perpendiculares a la acción elegida.
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_side),
            (perp2, p_side),
        ]

        transitions = []
        for move, prob in outcomes:
            next_state = (state[0] + move[0], state[1] + move[1])

            # Si el movimiento sale del grid o choca con una estantería,
            # el robot permanece en el mismo estado.
            if not self.is_valid_state(next_state):
                next_state = state

            transitions.append((next_state, prob))

        return transitions



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    # V_0(s) = 0 para todos los estados.
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        # Actualización sincrónica: V_{k+1} se calcula solo con V_k.
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                # Un terminal es absorbente: su valor es R(s).
                V_new[state] = grid.get_reward(state)
            else:
                # V_{k+1}(s) = R(s) + gamma * max_a sum_{s'} T(s,a,s') V_k(s')
                best_expected_value = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * best_expected_value
                )

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum_{s'} T(s,a,s') V(s')
    policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    # V_{k+1}^pi(s) = R(s) + gamma * sum_{s'} T(s,pi(s),s') V_k^pi(s')
    # (aquí NO hay max: la política ya eligió la acción)
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = (
                    grid.get_reward(state)
                    + grid.gamma * expected_next_value(grid, state, action, V)
                )

            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new

        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum_{s'} T(s,a,s') V^pi(s')
    new_policy = {}

    for state in grid.states():
        if grid.is_terminal(state):
            continue

        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria: RIGHT en todos los estados no terminales
    policy = {
        state: (0, 1)
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        # 2. evaluación
        V, eval_iterations = policy_evaluation(grid, policy, threshold=threshold)

        # 3. mejora
        new_policy = policy_improvement(grid, V)

        changed = sum(new_policy[state] != policy[state] for state in new_policy)
        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        # 4. repetir hasta estabilidad
        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 73, 'changed_actions': 19}, {'policy_iteration': 2, 'evaluation_sweeps': 61, 'changed_actions': 7}, {'policy_iteration': 3, 'evaluation_sweeps': 19, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +

In [7]:
# Utilidades de análisis (Parte 5)
from copy import deepcopy

NAMES = {(0, 5): "ENTREGA +10", (2, 2): "CARGA +2", (3, 5): "PELIGRO MORTAL -10"}


def solve(grid):
    """Resuelve el MDP con Value Iteration y devuelve (V*, pi*)."""
    V, _ = value_iteration(grid)
    return V, extract_policy(grid, V)


def terminal_probabilities(grid, policy, tol=1e-12, max_steps=5000):
    """Probabilidad de terminar en cada estado terminal siguiendo `policy` desde START (propagando la distribución)."""
    dist = {grid.start: 1.0}
    absorbed = {s: 0.0 for s in grid.terminal_states}
    for _ in range(max_steps):
        new = {}
        for s, p in dist.items():
            if grid.is_terminal(s):
                absorbed[s] += p
                continue
            for s2, q in grid.get_transition_probs(s, policy[s]):
                new[s2] = new.get(s2, 0.0) + p * q
        dist = new
        if sum(p for s, p in dist.items() if not grid.is_terminal(s)) < tol:
            for s, p in dist.items():
                if grid.is_terminal(s):
                    absorbed[s] += p
            break
    return absorbed


def intended_path(grid, policy, max_steps=50):
    """Camino «sin deslizamientos»: sigue la acción de la política (si choca, se queda) hasta un terminal."""
    s, path = grid.start, [grid.start]
    for _ in range(max_steps):
        if grid.is_terminal(s):
            break
        nxt = (s[0] + policy[s][0], s[1] + policy[s][1])
        if not grid.is_valid_state(nxt):
            nxt = s
        if nxt in path:          # bucle: la política no avanza
            path.append(nxt)
            break
        path.append(nxt)
        s = nxt
    return path


def final_target(grid, policy):
    path = intended_path(grid, policy)
    return NAMES.get(path[-1], "sin terminal (bucle)")


def report(title, grid):
    V, policy = solve(grid)
    probs = terminal_probabilities(grid, policy)
    print(f"=== {title} ===")
    print_policy(grid, policy)
    print(f"V*(START) = {V[grid.start]:+.3f}")
    print("Camino sin deslizamientos desde START:", " → ".join(map(str, intended_path(grid, policy))))
    print("Objetivo del camino:", final_target(grid, policy))
    print("Probabilidad de terminar en cada terminal:",
          {NAMES[s]: round(p, 3) for s, p in probs.items()})
    print()
    return V, policy


In [8]:
# Evidencia para las preguntas 1-5 (política óptima del mundo original)
V_base, pi_base = report("MUNDO ORIGINAL (living_reward=-1, gamma=0.9)", grid)

# Pregunta 3: ¿qué cambia si NINGÚN piso fuera resbaloso?
grid_dry = WarehouseMDP()
grid_dry.slippery_states = set()
V_dry, pi_dry = solve(grid_dry)

changed = sorted(s for s in pi_base if pi_base[s] != pi_dry[s])
print("Estados donde la política óptima cambia al quitar el piso resbaloso:")
for s in changed:
    print(f"  {s}: con resbaloso {ARROWS[pi_base[s]]}  |  sin resbaloso {ARROWS[pi_dry[s]]}")
print("\nPolítica si no hubiera piso resbaloso:")
print_policy(grid_dry, pi_dry)

# Pregunta 1/2: comparar los valores Q de las dos opciones desde las casillas clave
print("\nValores V* en las casillas clave:")
for s in [(0, 0), (0, 2), (1, 2), (1, 3), (1, 4)]:
    print(f"  V*{s} = {V_base[s]:+.3f}")


=== MUNDO ORIGINAL (living_reward=-1, gamma=0.9) ===
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
V*(START) = -2.575
Camino sin deslizamientos desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (2, 2)
Objetivo del camino: CARGA +2
Probabilidad de terminar en cada terminal: {'ENTREGA +10': 0.237, 'CARGA +2': 0.763, 'PELIGRO MORTAL -10': 0.0}

Estados donde la política óptima cambia al quitar el piso resbaloso:
  (1, 2): con resbaloso ↓  |  sin resbaloso →
  (3, 1): con resbaloso →  |  sin resbaloso ↑

Política si no hubiera piso resbaloso:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  ↑  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

Valores V* en las casillas clave:
  V*(0, 0) = -2.575
  V*(0, 2) = -0.652
  V*(1, 2) = +0.560
  V*(1, 3) = +2.104
  V*(1, 4) = +3.670


In [9]:
# Evidencia adicional para las preguntas 2 y 3

def q_values(grid, V, state):
    """Q(s,a) = R(s) + gamma * sum_{s'} T(s,a,s') V(s')"""
    return {
        ARROWS[a]: grid.get_reward(state) + grid.gamma * expected_next_value(grid, state, a, V)
        for a in grid.actions
    }

print("Q(s,a) en las casillas donde el piso resbaloso cambia la decisión:")
for s in [(1, 2), (3, 1)]:
    print(f"  Estado {s}")
    print("    con resbaloso :", {a: round(q, 3) for a, q in q_values(grid, V_base, s).items()})
    print("    sin resbaloso :", {a: round(q, 3) for a, q in q_values(grid_dry, V_dry, s).items()})

# Pregunta 2: ¿cuánto valdría START si solo existiera una de las dos opciones?
grid_only_delivery = WarehouseMDP()          # la estación de carga deja de ser terminal (queda como piso normal)
del grid_only_delivery.terminal_states[(2, 2)]
V_only_delivery, _ = solve(grid_only_delivery)

grid_only_charge = WarehouseMDP()            # la zona de entrega deja de ser terminal (queda como piso normal)
del grid_only_charge.terminal_states[(0, 5)]
V_only_charge, _ = solve(grid_only_charge)

print("\nV*(START) con ambas opciones      :", round(V_base[grid.start], 3))
print("V*(START) solo con la ENTREGA +10  :", round(V_only_delivery[grid.start], 3))
print("V*(START) solo con la CARGA +2     :", round(V_only_charge[grid.start], 3))


Q(s,a) en las casillas donde el piso resbaloso cambia la decisión:
  Estado (1, 2)
    con resbaloso : {'↑': -0.872, '↓': 0.56, '←': -0.455, '→': 0.379}
    sin resbaloso : {'↑': -1.245, '↓': 0.75, '←': -0.304, '→': 0.774}
  Estado (3, 1)
    con resbaloso : {'↑': -1.106, '↓': -4.206, '←': -2.612, '→': -0.731}
    sin resbaloso : {'↑': -0.54, '↓': -4.058, '←': -2.42, '→': -0.67}

V*(START) con ambas opciones      : -2.575
V*(START) solo con la ENTREGA +10  : -3.156
V*(START) solo con la CARGA +2     : -2.992



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas a las preguntas 1–5

*(Las cifras salen de las celdas de evidencia que se ejecutan justo antes.)*

**1. Desde `START`, ¿busca la entrega +10 o prefiere la carga +2?**
Prefiere la **estación de carga +2**. La política óptima desde `START` es →, →, ↓, ↓: (0,0) → (0,1) → (0,2) → (1,2) → (2,2). Siguiendo la política, el robot termina en la carga con probabilidad **0.763** y en la entrega con **0.237** (los deslizamientos en la casilla resbalosa (1,2) a veces lo empujan a (1,3), desde donde continúa hacia la entrega); la probabilidad de caer en el peligro mortal −10 es 0. $V^*(\text{START})=-2.575$.

**2. ¿Por qué una recompensa menor podría ser óptima?**
Porque lo que se maximiza no es la recompensa del terminal sino el **retorno descontado**, que incluye el costo por paso y el descuento $\gamma=0.9$. La carga está a 4 pasos. La entrega está a 7 pasos y, además, su único acceso pasa por la casilla de peligro (1,4) (−3): (0,4) y (1,5) solo son alcanzables a través de ella (o del terminal −10). Con esos costos, el +10 llega muy descontado. Si solo existiera una de las opciones, $V^*(\text{START})$ sería −2.992 con solo la carga y −3.156 con solo la entrega: la carga es la mejor opción individual. La política óptima mezcla ambas (−2.575), porque desde (1,2) se puede acabar en cualquiera de las dos según el deslizamiento.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
En **dos**: comparando con el mismo mundo sin piso resbaloso, la política óptima cambia en (1,2) y en (3,1).
- **(1,2)** (casilla resbalosa): sin resbalones → (hacia la entrega) y ↓ (hacia la carga) casi empatan ($Q=0.774$ frente a $0.750$); con resbalones → cae a 0.379 porque con probabilidad 0.2 se desliza hacia atrás, y ↓ queda en 0.560, así que el robot asegura la carga.
- **(3,1)**: sin resbalones sube (↑, $Q=-0.540$) pasando por la casilla resbalosa (2,1) para llegar a la carga; con resbalones esa casilla es mala ($Q(\uparrow)=-1.106$) y prefiere rodearla con → ($Q=-0.731$), entrando a la carga desde (3,2).

**4. ¿Qué papel cumple el costo por paso −1?**
Es una penalización por tardar: hace caras las rutas largas y empuja al robot a terminar pronto, incluso con una recompensa menor. Es el parámetro que decide entre carga (cercana) y entrega (lejana y con peligro): con −1 gana la carga; con −0.1 gana la entrega (experimento A); el cambio ocurre alrededor de −0.80 (bonus).

**5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?**
Porque la dinámica **depende del estado actual**: en piso normal el robot obedece con probabilidad 0.90 (0.05/0.05 de desviación) y en piso resbaloso con 0.60 (0.20/0.20). Por eso `get_transition_probs(state, action)` debe consultar `state in self.slippery_states` y usar las probabilidades de ese piso; en el GridWorld de clase las probabilidades (0.8/0.1/0.1) eran constantes y no hacía falta.

### Predicciones (escritas **antes** de ejecutar los experimentos)

Con `living_reward=-1` y $\gamma=0.9$ el robot desde START va hacia la **estación de carga** (+2), que está a 4 pasos, y no hacia la entrega (+10), que exige 7 pasos y cruzar el peligro −3.

- **Experimento A (`living_reward = -0.1`):** pasos mucho más baratos ⇒ ir hasta la entrega compensa. **Predicción:** desde START la política cambia a **intentar la ENTREGA +10**, cruzando la casilla de peligro (1,4).
- **Experimento B (resbaloso 0.60 → 0.40, desviaciones 0.30/0.30):** el piso resbaloso es mucho más impredecible. **Predicción:** el objetivo desde START sigue siendo la **carga**, pero cambian algunas acciones junto a las casillas amarillas (rodeándolas cuando se pueda) y bajan los valores de los estados cercanos.
- **Experimento C (`gamma = 0.99`):** el futuro se descuenta menos, así que el +10 «lejano» vale más. **Predicción:** la política desde START pasa a **valorar la entrega** (aunque con $\gamma=0.99$ también pesan más los costos acumulados por paso).
- **Bonus:** como con −1 gana la carga y con −0.1 gana la entrega, el umbral debe estar entre ambos; **predicción aproximada: `living_reward ≈ -0.6`**.

In [10]:
# Experimento A — menos costo por paso
grid_A = WarehouseMDP()
grid_A.living_reward = -0.1
V_A, pi_A = report("EXPERIMENTO A: living_reward = -0.1", grid_A)
print("Estados que cambian de acción respecto al original:",
      sorted(s for s in pi_base if pi_base[s] != pi_A[s]))

=== EXPERIMENTO A: living_reward = -0.1 ===
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
V*(START) = +1.649
Camino sin deslizamientos desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (1, 3) → (1, 4) → (1, 5) → (0, 5)
Objetivo del camino: ENTREGA +10
Probabilidad de terminar en cada terminal: {'ENTREGA +10': 0.748, 'CARGA +2': 0.252, 'PELIGRO MORTAL -10': 0.0}

Estados que cambian de acción respecto al original: [(1, 0), (1, 2), (3, 2)]


**Resultado A (`living_reward=-0.1`):** la predicción **se cumple**. Desde START el robot va ahora a la **entrega +10**: (0,0) → (0,1) → (0,2) → (1,2) → (1,3) → (1,4) → (1,5) → (0,5), cruzando el peligro (1,4). La probabilidad de terminar en la entrega sube de 0.237 a **0.748** y $V^*(\text{START})$ pasa de −2.575 a +1.649. Solo cambian de acción tres estados: (1,0), (1,2) y (3,2). Con pasos casi gratis, el +10 lejano compensa el peligro y la distancia.

In [11]:
# Experimento B — piso muy resbaloso: 0.60 -> 0.40 (el resto se reparte 0.30/0.30)
grid_B = WarehouseMDP()
grid_B.p_intended_slippery = 0.40
V_B, pi_B = report("EXPERIMENTO B: piso resbaloso con p(dirección elegida)=0.40", grid_B)
print("Estados que cambian de acción respecto al original:",
      sorted((s, ARROWS[pi_base[s]], ARROWS[pi_B[s]]) for s in pi_base if pi_base[s] != pi_B[s]))

=== EXPERIMENTO B: piso resbaloso con p(dirección elegida)=0.40 ===
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
V*(START) = -2.706
Camino sin deslizamientos desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (2, 2)
Objetivo del camino: CARGA +2
Probabilidad de terminar en cada terminal: {'ENTREGA +10': 0.406, 'CARGA +2': 0.594, 'PELIGRO MORTAL -10': 0.0}

Estados que cambian de acción respecto al original: []


**Resultado B (resbaloso 0.60 → 0.40):** la predicción se cumple **solo en parte**.
- Acerté en que el objetivo desde START sigue siendo la carga y en que los valores bajan ($V^*(\text{START})$: −2.575 → −2.706).
- **Me equivoqué** en que cambiarían acciones: **ninguna** acción cambia (0 estados), es decir, la política óptima es robusta a este cambio.
- Lo que sí cambia es el resultado probabilístico: como en (1,2) el robot se desliza más, la probabilidad de terminar en la entrega **sube** de 0.237 a 0.406 y la de la carga baja de 0.763 a 0.594.

In [12]:
# Experimento C — más paciencia
grid_C = WarehouseMDP()
grid_C.gamma = 0.99
V_C, pi_C = report("EXPERIMENTO C: gamma = 0.99", grid_C)
print("Estados que cambian de acción respecto al original:",
      sorted(s for s in pi_base if pi_base[s] != pi_C[s]))

=== EXPERIMENTO C: gamma = 0.99 ===
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 
V*(START) = -1.456
Camino sin deslizamientos desde START: (0, 0) → (0, 1) → (0, 2) → (1, 2) → (1, 3) → (1, 4) → (1, 5) → (0, 5)
Objetivo del camino: ENTREGA +10
Probabilidad de terminar en cada terminal: {'ENTREGA +10': 0.709, 'CARGA +2': 0.291, 'PELIGRO MORTAL -10': 0.0}

Estados que cambian de acción respecto al original: [(1, 2)]


**Resultado C (`gamma=0.99`):** la predicción **se cumple**. Con más paciencia el +10 pierde menos valor por la distancia y desde START el robot va a la **entrega** (mismo camino que en A, pasando por (1,2) → (1,3) → (1,4) → (1,5) → (0,5)). La probabilidad de terminar en la entrega es **0.709** y $V^*(\text{START})=-1.456$. Solo cambia una acción respecto al original: (1,2), que pasa de ↓ a →. Es decir, sí: la política valora más la recompensa +10 distante.

In [13]:
# Bonus — valor de living_reward donde la política desde START cambia entre CARGA y ENTREGA
def target_for(living_reward):
    g = WarehouseMDP()
    g.living_reward = living_reward
    _, pi = solve(g)
    return final_target(g, pi)

print("Barrido de living_reward:")
previous = None
for lr in [round(x, 2) for x in np.arange(-2.0, 0.01, 0.1)]:
    target = target_for(lr)
    flag = "  <-- cambio" if previous is not None and target != previous else ""
    print(f"  living_reward = {lr:+.2f} -> {target}{flag}")
    previous = target

# Búsqueda fina por bisección entre el último valor con CARGA y el primero con ENTREGA
lo, hi = -1.0, -0.1            # lo: CARGA, hi: ENTREGA (comprobado arriba)
assert target_for(lo) == "CARGA +2" and target_for(hi) == "ENTREGA +10"
for _ in range(30):
    mid = (lo + hi) / 2
    if target_for(mid) == "CARGA +2":
        lo = mid
    else:
        hi = mid
print(f"\nUmbral aproximado: living_reward ≈ {hi:.4f}  (carga si es menor, entrega si es mayor)")
BONUS_THRESHOLD = hi

Barrido de living_reward:
  living_reward = -2.00 -> CARGA +2
  living_reward = -1.90 -> CARGA +2
  living_reward = -1.80 -> CARGA +2


  living_reward = -1.70 -> CARGA +2
  living_reward = -1.60 -> CARGA +2
  living_reward = -1.50 -> CARGA +2
  living_reward = -1.40 -> CARGA +2
  living_reward = -1.30 -> CARGA +2


  living_reward = -1.20 -> CARGA +2
  living_reward = -1.10 -> CARGA +2
  living_reward = -1.00 -> CARGA +2
  living_reward = -0.90 -> CARGA +2


  living_reward = -0.80 -> CARGA +2
  living_reward = -0.70 -> ENTREGA +10  <-- cambio
  living_reward = -0.60 -> ENTREGA +10


  living_reward = -0.50 -> ENTREGA +10
  living_reward = -0.40 -> ENTREGA +10
  living_reward = -0.30 -> ENTREGA +10
  living_reward = -0.20 -> ENTREGA +10
  living_reward = -0.10 -> ENTREGA +10


  living_reward = +0.00 -> ENTREGA +10



Umbral aproximado: living_reward ≈ -0.7983  (carga si es menor, entrega si es mayor)


**Resultado del bonus:** el cambio entre ir a la carga y buscar la entrega ocurre con `living_reward ≈ -0.80` (la bisección da −0.798): por debajo de ese valor (pasos más caros) el robot va a la **carga +2**, por encima va a la **entrega +10**. El barrido en pasos de 0.1 lo confirma: carga hasta −0.80 y entrega desde −0.70. Mi predicción (≈ −0.6) quedó dentro del intervalo correcto, pero **desviada ≈0.2**.

*Criterio usado:* se considera que la política «va a» un terminal según el camino que resulta de seguir la acción elegida en cada estado (sin deslizamientos) desde START. La decisión que cambia en ese camino es la de (1,2): ↓ (carga) o → (entrega).